**<h1>Electric Line Extension - Analysis 1**
#### Descriptive Data
<i>Any question regarding the notebook, please contact Robert Ford or Ria Majumder<br>
    Last Updated: 08/13/2025 | Start Development: 08/13/2025</i>
* Utility / IOU Data from PG&E, SDG&E, and SCE for 2023, 2024, and Q1 2025
* Goals: 1. Clean excel files for public downloads. 


In [2]:
#Import Libraries and Packages
import pandas as pd
import numpy as np
import matplotlib as plt



In [183]:
pge_2023_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/PG&E Data_2023.xlsx", header=1)
sdge_2023_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/SDGE Data_2023.xlsx", header=1)
sce_2023_MFNC_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/Attachment A - 2024 Final SCE BE Annual Report.xlsx", header=4, sheet_name='Mixed-Fuel New Construction')
sce_2023_AENC_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/Attachment A - 2024 Final SCE BE Annual Report.xlsx", header=4, sheet_name='All Electric New Construction')
sce_2023_MFU_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/Attachment A - 2024 Final SCE BE Annual Report.xlsx", header=4, sheet_name='Mixed-Fuel Upgrades')
sce_2023_AEU_df = pd.read_excel("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/interim/Attachment A - 2024 Final SCE BE Annual Report.xlsx", header=4, sheet_name='All Electric Upgrades')



**CLEAN 1**

* For PG&E / SDG&E: Create dataframes for each sector ~ Residential, Nonresidential, and Combination (might include Mixed Use). 
    *   Add a 'Customer Class *' column & group by said column to create dataframe of all sectors
    *   Melt dataframes from wide to long using 'Month' & 'Customer Class *' as the IDs, variable name (columns) as 'type', values as 'counts'
    *   This results in a 624 x 4 dataframe for each IOU
* For SCE: 
    *   Using the individual sheets above, repeat steps for SCE
    *   Skip the summary rows, parse by month into blocks, concat blocks together and drop any repeated headers
    *   Group by 'Customer Class *', melt, and then concat results 
    *   This results in a 528 x 4 dataframe for each SCE sheet

In [ ]:
# Split the DataFrame into three separate DataFrames based on sector: residential, non-residential, and combined ~ drop NaN values
pge_2023_df_res = pge_2023_df.iloc[0:14].dropna(how='all').reset_index(drop=True)
pge_2023_df_nonres = pge_2023_df.iloc[17:31].dropna(how='all').reset_index(drop=True)
pge_2023_df_comb = pge_2023_df.iloc[34:50].dropna(how='all').reset_index(drop=True)
pge_2023_df_res
#pge_2023_df_nonres.reset_index(drop=True)
#pge_2023_df_comb.reset_index(drop=True)

In [ ]:
# Add a 'Customer Class *' column from SCE Annual Report to each sector dataframe
pge_2023_df_res['Customer Class *'] = 'Residential'
pge_2023_df_nonres['Customer Class *'] = 'Nonresidential'
pge_2023_df_comb['Customer Class *'] = 'Combination'

# Combine all three into one dataframe
pge_2023_df_all = pd.concat([pge_2023_df_res, pge_2023_df_nonres, pge_2023_df_comb], ignore_index=True)

# Move 'Customer Class *' to the front if you want
cols = ['Customer Class *'] + [col for col in pge_2023_df_all.columns if col != 'Customer Class *']
pge_2023_df_all = pge_2023_df_all[cols]

pge_2023_df_all

In [ ]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in pge_2023_df_all.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

sdge_2023_df_all_Melt = pd.concat(melted_blocks, ignore_index=True)
sdge_2023_df_all_Melt

In [ ]:
# Melt the DataFrames to long format
pge_2023_df_recomb = pge_2023_df_comb.melt(id_vars=['Month'], var_name='Type', value_name='Count')
pge_2023_df_renonres = pge_2023_df_nonres.melt(id_vars=['Month'], var_name='Type', value_name='Count')
pge_2023_df_reres = pge_2023_df_res.melt(id_vars=['Month'], var_name='Type', value_name='Count')
pge_2023_df_reres
#pge_2023_df_recomb
#pge_2023_df_renonres


In [ ]:
# Split the DataFrame into three separate DataFrames based on sector: residential, non-residential, and combined ~ drop NaN values
# The sdge_2023_df_comb dataframe is only residential and non-residential, NOT mixed use.
sdge_2023_df_res = sdge_2023_df.iloc[0:14].dropna(how='all').reset_index(drop=True)
sdge_2023_df_nonres = sdge_2023_df.iloc[18:32].dropna(how='all').reset_index(drop=True)
sdge_2023_df_comb = sdge_2023_df.iloc[36:50].dropna(how='all').reset_index(drop=True)
sdge_2023_df_comb

In [ ]:
# Add a 'Customer Class *' column from SCE Annual Report to each sector dataframe
sdge_2023_df_res['Customer Class *'] = 'Residential'
sdge_2023_df_nonres['Customer Class *'] = 'Nonresidential'
sdge_2023_df_comb['Customer Class *'] = 'Combination'

# Combine all three into one dataframe
sdge_2023_df_all = pd.concat([sdge_2023_df_res, sdge_2023_df_nonres, sdge_2023_df_comb], ignore_index=True)

# Move 'Customer Class *' to the front if you want
cols = ['Customer Class *'] + [col for col in sdge_2023_df_all.columns if col != 'Customer Class *']
sdge_2023_df_all = sdge_2023_df_all[cols]

sdge_2023_df_all

In [ ]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in sdge_2023_df_all.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

sdge_2023_df_all_Melt = pd.concat(melted_blocks, ignore_index=True)
sdge_2023_df_all_Melt

In [184]:
# Prepare each individual sheet in the SCE 2023 data for melting
df = sce_2023_MFNC_df.copy()

monthly_blocks = []
row = 4  # Start after summary

while row < len(df):
    # Get the month label from the first cell of the row
    month_cell = df.iloc[row, 0]
    if pd.isna(month_cell):
        row += 1
        continue  # skip blank rows

    # Try to parse the month label
    try:
        month_label = pd.to_datetime(month_cell).strftime('%b %y')
    except Exception:
        row += 1
        continue  # skip if not a date

    # The next 5 rows are the data for this month (skip the header row)
    data_block = df.iloc[row+1:row+6].copy()
    data_block['Month'] = month_label
    monthly_blocks.append(data_block)

    # Move to the next block (skip the 5 data rows + 1 blank row)
    row += 6

# Combine all blocks
sce_2023_MFNC_clean = pd.concat(monthly_blocks, ignore_index=True)

# Optionally, drop rows where all columns except 'Month' are NaN (in case of trailing blanks)
sce_2023_MFNC_clean = sce_2023_MFNC_clean.dropna(subset=[col for col in sce_2023_MFNC_clean.columns if col != 'Month'], how='all')

# Move 'Month' to the front
cols = ['Month'] + [col for col in sce_2023_MFNC_clean.columns if col != 'Month']
sce_2023_MFNC_clean = sce_2023_MFNC_clean[cols]

# Drop every 5th row to match the expected number of rows
sce_2023_MFNC_clean = sce_2023_MFNC_clean.drop(sce_2023_MFNC_clean.index[::5]).reset_index(drop=True)

sce_2023_MFNC_clean

,Month,Customer Class *,Total Discounts (Non-Exempted Projects),Total Discounts (Exempted projects),Total Allowances (Non-Exempted Projects),Total Allowances (Exempted projects),Total Refund Payments Provided to Builders (Non-Exempted Projects),Total Refund Payments Provided to Builders (Exempted projects),Total Estimated Non-Refundable,Total Estimated Refundable,Total Electric Line Extension Requests Received (Applications),Total Electric Line Extensions Energized,Total Electric Line Extension Applications for Applicant Install
0,Jan 23,Residential,361982.8,0,1250226.91,0,2252901.63,0,467806.72,1912673.28,556,345,32
1,Jan 23,Industrial,0,0,0,0,0,0,0,0,0,0,0
2,Jan 23,Commercial,410316.32,0,2843419.42,0,396422.54,0,652880.76,16427.11,307,159,17
3,Jan 23,Agriculture,9098.22,0,154834.34,0,0,0,1092.75,0,16,10,0
4,Feb 24,Residential,454249.66,0,1080424.47,0,2985830.88,0,550610.99,1591659.09,603,341,11
5,Feb 24,Industrial,0,0,0,0,0,0,0,0,0,0,0
6,Feb 24,Commercial,233258.66,0,2531623.19,0,1329385.67,0,346507.42,0,300,120,11
7,Feb 24,Agriculture,8625.36,0,61999.47,0,1738.36,0,1308.15,0,14,7,0
8,Mar 23,Residential,409785.02,0,1918241.07,0,3864853.56,0,604846.16,1728677.93,693,480,24
9,Mar 23,Industrial,0,0,0,0,0,0,0,0,0,0,0


In [185]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in sce_2023_MFNC_clean.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

sce_2023_MFNC_melted = pd.concat(melted_blocks, ignore_index=True)
sce_2023_MFNC_melted

,Month,Customer Class *,Type,Count
0,Jan 23,Agriculture,Total Discounts (Non-Exempted Projects),9098.22
1,Feb 24,Agriculture,Total Discounts (Non-Exempted Projects),8625.36
2,Mar 23,Agriculture,Total Discounts (Non-Exempted Projects),3208.13
3,Apr 23,Agriculture,Total Discounts (Non-Exempted Projects),881.12
4,May 23,Agriculture,Total Discounts (Non-Exempted Projects),0
...,...,...,...,...
523,Aug 23,Residential,Total Electric Line Extension Applications for...,13
524,Sep 23,Residential,Total Electric Line Extension Applications for...,21
525,Oct 23,Residential,Total Electric Line Extension Applications for...,32
526,Nov 23,Residential,Total Electric Line Extension Applications for...,37


In [186]:
# Prepare each individual sheet in the SCE 2023 data for melting
df2 = sce_2023_AENC_df.copy()

monthly_blocks = []
row = 4  # Start after summary

while row < len(df2):
    # Get the month label from the first cell of the row
    month_cell = df2.iloc[row, 0]
    if pd.isna(month_cell):
        row += 1
        continue  # skip blank rows

    # Try to parse the month label
    try:
        month_label = pd.to_datetime(month_cell).strftime('%b %y')
    except Exception:
        row += 1
        continue  # skip if not a date

    # The next 5 rows are the data for this month (skip the header row)
    data_block = df2.iloc[row+1:row+6].copy()
    data_block['Month'] = month_label
    monthly_blocks.append(data_block)

    # Move to the next block (skip the 5 data rows + 1 blank row)
    row += 6

# Combine all blocks
sce_2023_AENC_clean = pd.concat(monthly_blocks, ignore_index=True)

# Optionally, drop rows where all columns except 'Month' are NaN (in case of trailing blanks)
sce_2023_AENC_clean = sce_2023_AENC_clean.dropna(subset=[col for col in sce_2023_AENC_clean.columns if col != 'Month'], how='all')

# Move 'Month' to the front
cols = ['Month'] + [col for col in sce_2023_AENC_clean.columns if col != 'Month']
sce_2023_AENC_clean = sce_2023_AENC_clean[cols]

# Drop every 5th row to match the expected number of rows
sce_2023_AENC_clean = sce_2023_AENC_clean.drop(sce_2023_AENC_clean.index[::5]).reset_index(drop=True)

sce_2023_AENC_clean

,Month,Customer Class *,Total Discounts (Non-Exempted Projects),Total Discounts (Exempted projects),Total Allowances (Non-Exempted Projects),Total Allowances (Exempted projects),Total Refund Payments Provided to Builders (Non-Exempted Projects),Total Refund Payments Provided to Builders (Exempted projects),Total Estimated Non-Refundable,Total Estimated Refundable,Total Electric Line Extension Requests Received (Applications),Total Electric Line Extensions Energized,Total Electric Line Extension Applications for Applicant Install
0,Jan 23,Residential,65107.43,0,23992.77,0,0,0,24884.58,0,21,9,2
1,Jan 23,Industrial,0,0,0,0,0,0,0,0,0,0,0
2,Jan 23,Commercial,0,0,692106.78,0,0,0,46142.46,1499.97,23,14,0
3,Jan 23,Agriculture,0,0,0,0,0,0,0,0,2,0,0
4,Feb 24,Residential,0,0,20860.32,0,0,0,0,22205.04,23,9,0
5,Feb 24,Industrial,0,0,0,0,0,0,0,0,0,0,0
6,Feb 24,Commercial,0,0,1331354.53,0,0,0,80372.23,0,20,13,0
7,Feb 24,Agriculture,0,0,0,0,0,0,0,0,0,0,0
8,Mar 23,Residential,41286.15,0,60085.18,0,0,0,17435.74,0,38,13,1
9,Mar 23,Industrial,0,0,0,0,0,0,0,0,0,0,0


In [187]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in sce_2023_AENC_clean.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

sce_2023_AENC_melted = pd.concat(melted_blocks, ignore_index=True)
sce_2023_AENC_melted

,Month,Customer Class *,Type,Count
0,Jan 23,Agriculture,Total Discounts (Non-Exempted Projects),0
1,Feb 24,Agriculture,Total Discounts (Non-Exempted Projects),0
2,Mar 23,Agriculture,Total Discounts (Non-Exempted Projects),0
3,Apr 23,Agriculture,Total Discounts (Non-Exempted Projects),0
4,May 23,Agriculture,Total Discounts (Non-Exempted Projects),8783.38
...,...,...,...,...
523,Aug 23,Residential,Total Electric Line Extension Applications for...,1
524,Sep 23,Residential,Total Electric Line Extension Applications for...,1
525,Oct 23,Residential,Total Electric Line Extension Applications for...,0
526,Nov 23,Residential,Total Electric Line Extension Applications for...,0


In [188]:
# Prepare each individual sheet in the SCE 2023 data for melting
df3 = sce_2023_MFU_df.copy()

monthly_blocks = []
row = 4  # Start after summary

while row < len(df3):
    # Get the month label from the first cell of the row
    month_cell = df3.iloc[row, 0]
    if pd.isna(month_cell):
        row += 1
        continue  # skip blank rows

    # Try to parse the month label
    try:
        month_label = pd.to_datetime(month_cell).strftime('%b %y')
    except Exception:
        row += 1
        continue  # skip if not a date

    # The next 5 rows are the data for this month (skip the header row)
    data_block = df3.iloc[row+1:row+6].copy()
    data_block['Month'] = month_label
    monthly_blocks.append(data_block)

    # Move to the next block (skip the 5 data rows + 1 blank row)
    row += 6

# Combine all blocks
sce_2023_MFU_clean = pd.concat(monthly_blocks, ignore_index=True)

# Optionally, drop rows where all columns except 'Month' are NaN (in case of trailing blanks)
sce_2023_MFU_clean = sce_2023_MFU_clean.dropna(subset=[col for col in sce_2023_MFU_clean.columns if col != 'Month'], how='all')

# Move 'Month' to the front
cols = ['Month'] + [col for col in sce_2023_MFU_clean.columns if col != 'Month']
sce_2023_MFU_clean = sce_2023_MFU_clean[cols]

# Drop every 5th row to match the expected number of rows
sce_2023_MFU_clean = sce_2023_MFU_clean.drop(sce_2023_MFU_clean.index[::5]).reset_index(drop=True)

sce_2023_MFU_clean

,Month,Customer Class *,Total Discounts (Non-Exempted Projects),Total Discounts (Exempted projects),Total Allowances (Non-Exempted Projects),Total Allowances (Exempted projects),Total Refund Payments Provided to Builders (Non-Exempted Projects),Total Refund Payments Provided to Builders (Exempted projects),Total Estimated Non-Refundable,Total Estimated Refundable,Total Electric Line Extension Requests Received (Applications),Total Electric Line Extensions Energized,Total Electric Line Extension Applications for Applicant Install
0,Jan 23,Residential,0,0,288372.5,0,0,0,0,0,816,415,0
1,Jan 23,Industrial,0,0,0,0,0,0,0,0,0,0,0
2,Jan 23,Commercial,0,0,434382.0,0,0,0,0,0,63,28,0
3,Jan 23,Agriculture,0,0,91770.09,0,0,0,0,0,15,9,0
4,Feb 24,Residential,0,0,274458.77,0,0,0,0,0,872,411,0
5,Feb 24,Industrial,0,0,0,0,0,0,0,0,0,0,0
6,Feb 24,Commercial,0,0,646380.56,0,0,0,0,0,72,33,0
7,Feb 24,Agriculture,0,0,281569.33,0,0,0,0,0,18,10,0
8,Mar 23,Residential,0,0,348657.16,0,0,0,0,0,979,519,0
9,Mar 23,Industrial,0,0,0,0,0,0,0,0,0,0,0


In [189]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in sce_2023_MFU_clean.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

sce_2023_MFU_melted = pd.concat(melted_blocks, ignore_index=True)
sce_2023_MFU_melted

,Month,Customer Class *,Type,Count
0,Jan 23,Agriculture,Total Discounts (Non-Exempted Projects),0
1,Feb 24,Agriculture,Total Discounts (Non-Exempted Projects),0
2,Mar 23,Agriculture,Total Discounts (Non-Exempted Projects),0
3,Apr 23,Agriculture,Total Discounts (Non-Exempted Projects),0
4,May 23,Agriculture,Total Discounts (Non-Exempted Projects),0
...,...,...,...,...
523,Aug 23,Residential,Total Electric Line Extension Applications for...,0
524,Sep 23,Residential,Total Electric Line Extension Applications for...,0
525,Oct 23,Residential,Total Electric Line Extension Applications for...,0
526,Nov 23,Residential,Total Electric Line Extension Applications for...,1


In [190]:
# Prepare each individual sheet in the SCE 2023 data for melting
df4 = sce_2023_AEU_df.copy()

monthly_blocks = []
row = 4  # Start after summary

while row < len(df4):
    # Get the month label from the first cell of the row
    month_cell = df4.iloc[row, 0]
    if pd.isna(month_cell):
        row += 1
        continue  # skip blank rows

    # Try to parse the month label
    try:
        month_label = pd.to_datetime(month_cell).strftime('%b %y')
    except Exception:
        row += 1
        continue  # skip if not a date

    # The next 5 rows are the data for this month (skip the header row)
    data_block = df4.iloc[row+1:row+6].copy()
    data_block['Month'] = month_label
    monthly_blocks.append(data_block)

    # Move to the next block (skip the 5 data rows + 1 blank row)
    row += 6

# Combine all blocks
sce_2023_AEU_clean = pd.concat(monthly_blocks, ignore_index=True)

# Optionally, drop rows where all columns except 'Month' are NaN (in case of trailing blanks)
sce_2023_AEU_clean = sce_2023_AEU_clean.dropna(subset=[col for col in sce_2023_AEU_clean.columns if col != 'Month'], how='all')

# Move 'Month' to the front
cols = ['Month'] + [col for col in sce_2023_AEU_clean.columns if col != 'Month']
sce_2023_AEU_clean = sce_2023_AEU_clean[cols]

# Drop every 5th row to match the expected number of rows
sce_2023_AEU_clean = sce_2023_AEU_clean.drop(sce_2023_AEU_clean.index[::5]).reset_index(drop=True)

sce_2023_AEU_clean

,Month,Customer Class *,Total Discounts (Non-Exempted Projects),Total Discounts (Exempted projects),Total Allowances (Non-Exempted Projects),Total Allowances (Exempted projects),Total Refund Payments Provided to Builders (Non-Exempted Projects),Total Refund Payments Provided to Builders (Exempted projects),Total Estimated Non-Refundable,Total Estimated Refundable,Total Electric Line Extension Requests Received (Applications),Total Electric Line Extensions Energized,Total Electric Line Extension Applications for Applicant Install
0,Jan 23,Residential,0,0,6485.06,0,0,0,0,0,5,3,0
1,Jan 23,Industrial,0,0,0,0,0,0,0,0,0,0,0
2,Jan 23,Commercial,0,0,128253.7,0,0,0,0,0,0,4,0
3,Jan 23,Agriculture,0,0,0,0,0,0,0,0,0,0,0
4,Feb 24,Residential,0,0,539.61,0,0,0,0,0,3,1,0
5,Feb 24,Industrial,0,0,0,0,0,0,0,0,0,0,0
6,Feb 24,Commercial,0,0,0,0,0,0,0,0,0,0,0
7,Feb 24,Agriculture,0,0,0,0,0,0,0,0,0,0,0
8,Mar 23,Residential,0,0,4434.74,0,0,0,0,0,7,4,0
9,Mar 23,Industrial,0,0,0,0,0,0,0,0,0,0,0


In [191]:
# Group by 'Customer Class *' and melt each group, then concatenate the results
melted_blocks = []
for name, group in sce_2023_AEU_clean.groupby('Customer Class *'):
    melted = group.melt(id_vars=['Month', 'Customer Class *'], var_name='Type', value_name='Count')
    melted_blocks.append(melted)

sce_2023_AEU_melted = pd.concat(melted_blocks, ignore_index=True)
sce_2023_AEU_melted

,Month,Customer Class *,Type,Count
0,Jan 23,Agriculture,Total Discounts (Non-Exempted Projects),0
1,Feb 24,Agriculture,Total Discounts (Non-Exempted Projects),0
2,Mar 23,Agriculture,Total Discounts (Non-Exempted Projects),0
3,Apr 23,Agriculture,Total Discounts (Non-Exempted Projects),0
4,May 23,Agriculture,Total Discounts (Non-Exempted Projects),0
...,...,...,...,...
523,Aug 23,Residential,Total Electric Line Extension Applications for...,0
524,Sep 23,Residential,Total Electric Line Extension Applications for...,0
525,Oct 23,Residential,Total Electric Line Extension Applications for...,0
526,Nov 23,Residential,Total Electric Line Extension Applications for...,0
